# Resumen — Inferencia Estadística y Tests de Hipótesis

> **Para el TP6 de Inferencia Estadística**  
> Estimación, intervalos de confianza, test t, ANOVA, Mann-Whitney, Kruskal-Wallis, chi-cuadrado y Shapiro-Wilk.


In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt

np.random.seed(42)
n = 120
df = pd.DataFrame({
    'goles_local'    : np.random.poisson(lam=1.6, size=n),
    'goles_visitante': np.random.poisson(lam=1.1, size=n),
    'goles_t1'       : np.random.poisson(lam=1.2, size=n),
    'goles_t2'       : np.random.poisson(lam=1.5, size=n),
    'liga'           : np.random.choice(['LaLiga', 'Premier', 'SerieA'], size=n),
    'resultado'      : np.random.choice(['Ganó Local', 'Empate', 'Ganó Visitante'],
                                         size=n, p=[0.45, 0.25, 0.30])
})
print('Dataset listo. Shape:', df.shape)


---
## 1. La lógica de los tests de hipótesis

Siempre seguimos el mismo esquema:

```
1. H0 (hipótesis nula): 'no hay efecto / no hay diferencia'
2. H1 (hipótesis alternativa): 'sí hay efecto / sí hay diferencia'
3. Calculamos el estadístico del test
4. Obtenemos el p-valor
5. Si p-valor < α (usualmente 0.05) → RECHAZAMOS H0
   Si p-valor >= α               → NO rechazamos H0 (no es lo mismo que 'H0 es verdadera')
```

**El p-valor** es la probabilidad de obtener datos tan extremos como los observados, asumiendo que H0 es verdadera. Un p-valor pequeño = los datos son poco compatibles con H0.

> **Error tipo I (α):** rechazar H0 cuando era verdadera. Lo controlamos con α = 0.05.  
> **Error tipo II (β):** no rechazar H0 cuando era falsa. Se minimiza con más datos (n grande).


---
## 2. Estimación puntual e Intervalo de Confianza (IC)


In [ ]:
# Estimación puntual
media = df['goles_local'].mean()
print(f'Estimación puntual de la media: {media:.4f}')

# Intervalo de confianza al 95%
n_obs = len(df['goles_local'])
sem   = df['goles_local'].std(ddof=1) / np.sqrt(n_obs)
ic    = stats.t.interval(0.95, df=n_obs-1, loc=media, scale=sem)

print(f'IC 95%: ({ic[0]:.4f}, {ic[1]:.4f})')
print()
print('Interpretación: con 95% de confianza el promedio real está dentro de ese rango.')
print('Si repitiéramos el muestreo 100 veces, ~95 ICs contendrían el valor verdadero.')


---
## 3. Test t — tres variantes

| Variante | Cuándo usar | Función |
|---|---|---|
| **1 muestra** | Comparar una muestra contra un valor de referencia | `stats.ttest_1samp(data, popmean)` |
| **2 muestras independientes** | Comparar dos grupos distintos | `stats.ttest_ind(grupo1, grupo2)` |
| **Pareado** | Comparar el mismo grupo antes/después | `stats.ttest_rel(antes, despues)` |

**Supuesto:** los datos deben ser aproximadamente normales (o n > 30 por el teorema central del límite).


In [ ]:
# Test t - 1 muestra: ¿la media de goles locales es igual a 2?
t1, p1 = stats.ttest_1samp(df['goles_local'], popmean=2.0)
print(f'Test 1 muestra (mu=2.0): t={t1:.3f}, p={p1:.4f}')
print(f'  => {"RECHAZA H0" if p1 < 0.05 else "No rechaza H0"}')

# Test t - 2 muestras: ¿local vs visitante?
t2, p2 = stats.ttest_ind(df['goles_local'], df['goles_visitante'])
print(f'\nTest 2 muestras (local vs visitante): t={t2:.3f}, p={p2:.4f}')
print(f'  => {"RECHAZA H0" if p2 < 0.05 else "No rechaza H0"}')

# Test t - pareado: ¿mejoró el equipo con el nuevo entrenador?
t3, p3 = stats.ttest_rel(df['goles_t2'], df['goles_t1'])
print(f'\nTest pareado (T2 vs T1): t={t3:.3f}, p={p3:.4f}')
print(f'  => {"RECHAZA H0" if p3 < 0.05 else "No rechaza H0"}')


---
## 4. ANOVA — comparar MÁS de dos grupos

El test t solo compara **dos** grupos. Para comparar **tres o más** usamos ANOVA.

- **H0:** todas las medias son iguales (μ1 = μ2 = μ3 = ...)
- **H1:** al menos una media difiere
- Usa el **estadístico F** (razón de varianzas)
- Si p < 0.05: hay diferencia, pero ANOVA no dice **cuáles** grupos difieren → necesitamos tests post-hoc

**Supuesto:** normalidad y homocedasticidad (varianzas similares entre grupos).


In [ ]:
laliga  = df[df['liga'] == 'LaLiga']['goles_local']
premier = df[df['liga'] == 'Premier']['goles_local']
seriea  = df[df['liga'] == 'SerieA']['goles_local']

f, p = stats.f_oneway(laliga, premier, seriea)
print(f'ANOVA — F = {f:.4f}, p = {p:.4f}')
print(f'  => {"RECHAZA H0: al menos una liga difiere" if p < 0.05 else "No rechaza H0: ligas similares"}')

print(f'\nMedias: LaLiga={laliga.mean():.2f}  Premier={premier.mean():.2f}  SerieA={seriea.mean():.2f}')


---
## 5. Tests No Paramétricos

Se usan cuando los datos **no son normales** (como datos de conteo, ordinales, o muy asimétricos).

| Test paramétrico | Equivalente no paramétrico | Caso de uso |
|---|---|---|
| Test t 2 muestras | **Mann-Whitney U** | Comparar 2 grupos independientes |
| ANOVA | **Kruskal-Wallis** | Comparar 3+ grupos independientes |

En lugar de comparar **medias**, estos tests comparan las **distribuciones** usando rangos.


In [ ]:
# Mann-Whitney: LaLiga vs Premier
u, p_mw = stats.mannwhitneyu(laliga, premier, alternative='two-sided')
print(f'Mann-Whitney (LaLiga vs Premier): U={u:.1f}, p={p_mw:.4f}')
print(f'  => {"RECHAZA H0" if p_mw < 0.05 else "No rechaza H0"}')

# Kruskal-Wallis: las tres ligas
h, p_kw = stats.kruskal(laliga, premier, seriea)
print(f'\nKruskal-Wallis (3 ligas): H={h:.4f}, p={p_kw:.4f}')
print(f'  => {"RECHAZA H0" if p_kw < 0.05 else "No rechaza H0"}')


---
## 6. Chi-Cuadrado — variables categóricas

Cuando queremos saber si **dos variables categóricas están asociadas** entre sí.

- **H0:** las variables son **independientes** (no hay asociación)
- **H1:** las variables **no son independientes** (hay asociación)
- Se construye una **tabla de contingencia** y se compara con los valores esperados bajo independencia


In [ ]:
tabla = pd.crosstab(df['liga'], df['resultado'])
print('Tabla de contingencia:')
print(tabla)

chi2, p_chi, dof, esperado = stats.chi2_contingency(tabla)
print(f'\nchi2 = {chi2:.4f},  gl = {dof},  p = {p_chi:.4f}')
print(f'=> {"RECHAZA H0: liga asociada al resultado" if p_chi < 0.05 else "No rechaza H0: son independientes"}')


---
## 7. Shapiro-Wilk — prueba de normalidad

Antes de usar tests paramétricos (t, ANOVA), es bueno verificar si los datos son normales.

- **H0:** los datos siguen una distribución normal
- **H1:** los datos NO siguen una distribución normal
- Si p < 0.05 → los datos **no son normales** → usar tests no paramétricos

**Limitación:** con muestras muy grandes (n > 2000), casi siempre rechaza H0 aunque la desviación de la normalidad sea mínima e irrelevante.


In [ ]:
w, p_shapiro = stats.shapiro(df['goles_local'])
print(f'Shapiro-Wilk: W={w:.4f}, p={p_shapiro:.4f}')
print(f'=> {"NO son normales" if p_shapiro < 0.05 else "Compatibles con normalidad"}')

print()
print('Los goles siguen distribución Poisson (conteos discretos), NO Normal.')
print('Por eso se espera rechazar normalidad → usar Mann-Whitney / Kruskal-Wallis.')


---
## 8. Mapa de decisión: ¿qué test usar?

```
¿Cuántos grupos comparo?
│
├─ 1 grupo vs valor de referencia ─────────────────► Test t 1 muestra
│
├─ 2 grupos ─── ¿Independientes o pareados?
│   │
│   ├─ Independientes ── ¿Datos normales?
│   │   ├─ Sí ──────────────────────────────────────► Test t 2 muestras
│   │   └─ No ──────────────────────────────────────► Mann-Whitney U
│   │
│   └─ Pareados (mismo grupo antes/después) ─────────► Test t pareado
│
└─ 3+ grupos ── ¿Datos normales?
    ├─ Sí ──────────────────────────────────────────► ANOVA (f_oneway)
    └─ No ──────────────────────────────────────────► Kruskal-Wallis

¿Variables categóricas? ─────────────────────────────► Chi-cuadrado
¿Verificar normalidad? ──────────────────────────────► Shapiro-Wilk
```

> **Tip de examen:** si los datos son **goles, precios, ingresos, conteos** o cualquier cosa claramente no normal → usá directamente los tests **no paramétricos** sin necesidad de verificar.
